In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 9
fig_height = 6
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWl0aHVubWFuaXZhbm5hbi9wcm9qZWN0cy9iZW5jaG1hcmtpbmdfbG9zc19mdW5jdGlvbnNfZWNnX3JlY29uc3RydWN0aW9uL2Jvb2s='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/usr/lib/python3.12/importlib/_bootstrap.py": 1781873160.0, "/usr/lib/python3.12/importlib/_bootstrap_external.py": 1781873160.0, "/usr/lib/python3.12/zipimport.py": 1781873160.0, "/usr/lib/python3.12/codecs.py": 1781873160.0, "/usr/lib/python3.12/encodings/aliases.py": 1781873160.0, "/usr/lib/python3.12/encodings/__init__.py": 1781873160.0, "/usr/lib/python3.12/encodings/utf_8.py": 1781873160.0, "/usr/lib/python3.12/abc.py": 1781873160.0, "/usr/lib/python3.12/io.py": 1781873160.0, "/usr/lib/python3.12/stat.py": 1781873160.0, "/usr/lib/python3.12/_collections_abc.py": 1781873160.0, "/usr/lib/python3.12/genericpath.py": 1781873160.0, "/usr/lib/python3.12/posixpath.py": 1781873160.0, "/usr/lib/python3.12/os.py": 1781873160.0, "/usr/lib/python3.12/_sitebuiltins.py": 1781873160.0, "/usr/lib/python3.12/__future__.py": 1781873160.0, "/usr/lib/python3.12/warnings.py": 1781873160.0, "/usr/lib/python3.12/importlib/__init__.py": 1781873160.0, "/usr/lib/python3.12/importlib/machinery.py": 17818

In [2]:
#| label: qtc-sensitivity-plot
import numpy as np
import plotly.graph_objects as go

hr_range = np.linspace(40, 140, 100)
rr_seconds = 60.0 / hr_range
qt_ms = 400.0  # Constant 400 ms QT interval

qtc_bazett = qt_ms / np.sqrt(rr_seconds)
qtc_fridericia = qt_ms / (rr_seconds ** (1/3))
qtc_framingham = qt_ms + 154.0 * (1.0 - rr_seconds)
qtc_hodges = qt_ms + 1.75 * (hr_range - 60.0)

fig_qtc = go.Figure()
fig_qtc.add_trace(go.Scatter(x=hr_range, y=qtc_bazett, mode='lines', line=dict(color='#f43f5e', width=2), name='Bazett (Overcorrects at High HR)'))
fig_qtc.add_trace(go.Scatter(x=hr_range, y=qtc_fridericia, mode='lines', line=dict(color='#38bdf8', width=2), name='Fridericia (Recommended Standard)'))
fig_qtc.add_trace(go.Scatter(x=hr_range, y=qtc_framingham, mode='lines', line=dict(color='#10b981', width=2), name='Framingham Linear Correction'))
fig_qtc.add_trace(go.Scatter(x=hr_range, y=qtc_hodges, mode='lines', line=dict(color='#fbbf24', width=2), name='Hodges Linear Correction'))

fig_qtc.update_layout(
    title="QTc Correction Formula Heart Rate Sensitivity (Fixed 400 ms QT)",
    xaxis_title="Heart Rate (bpm)",
    yaxis_title="Corrected QTc (ms)",
    template="plotly_dark",
    height=450, margin=dict(l=20, r=20, t=50, b=20)
)
fig_qtc.show()

In [3]:
import neurokit2 as nk
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# 1. Simulate one 5-second ECG-like trace at 500 Hz
sampling_rate = 500
ecg_signal = nk.ecg_simulate(duration=5, sampling_rate=sampling_rate, heart_rate=72)

# 2. Clean signal using zero-phase Butterworth bandpass filter
ecg_cleaned = nk.ecg_clean(ecg_signal, sampling_rate=sampling_rate, method="neurokit")

# 3. Locate R-peaks
_, rpeaks = nk.ecg_peaks(ecg_cleaned, sampling_rate=sampling_rate)

# 4. Continuous Wavelet Transform (CWT) Delineation for P, Q, S, T boundaries
signals, waves = nk.ecg_delineate(ecg_cleaned, rpeaks, sampling_rate=sampling_rate, method="cwt")

# Generate time vector in seconds
t = np.linspace(0, len(ecg_cleaned) / sampling_rate, len(ecg_cleaned))

# Create Plotly Interactive Delineation Viewer
fig_del = go.Figure()

# Cleaned ECG trace
fig_del.add_trace(go.Scatter(
    x=t, y=ecg_cleaned, mode='lines', 
    name='Cleaned synthetic trace',
    line=dict(color='#38bdf8', width=2)
))

# Helper to plot fiducial markers
def add_fiducial(indices, color, symbol, name):
    valid_idx = [int(i) for i in indices if not np.isnan(i)]
    if len(valid_idx) > 0:
        fig_del.add_trace(go.Scatter(
            x=t[valid_idx], y=ecg_cleaned[valid_idx],
            mode='markers', 
            marker=dict(color=color, size=9, symbol=symbol, line=dict(width=1, color='white')),
            name=name
        ))

# Add P, R, and T wave markers
add_fiducial(rpeaks['ECG_R_Peaks'], '#f43f5e', 'star', 'R-Peak')
add_fiducial(waves['ECG_P_Onsets'], '#a855f7', 'triangle-right', 'P-Onset')
add_fiducial(waves['ECG_P_Offsets'], '#a855f7', 'triangle-left', 'P-Offset')
add_fiducial(waves['ECG_T_Onsets'], '#10b981', 'triangle-right', 'T-Onset')
add_fiducial(waves['ECG_T_Offsets'], '#10b981', 'triangle-left', 'T-Offset')

fig_del.update_layout(
    title="DIDACTIC SIMULATION — Automated CWT Delineation",
    xaxis_title="Time (seconds)",
    yaxis_title="Simulated amplitude (arbitrary units)",
    template="plotly_dark",
    height=450, margin=dict(l=20, r=20, t=40, b=20)
)
fig_del.show()

In [4]:
def calculate_qtc_variations(qt_ms, rr_ms):
    """
    Computes QTc across Bazett, Fridericia, Framingham, and Hodges formulas.
    
    Args:
        qt_ms (float): Uncorrected QT interval in milliseconds.
        rr_ms (float): R-R interval in milliseconds.
    """
    qt_sec = qt_ms / 1000.0
    rr_sec = rr_ms / 1000.0
    hr = 60.0 / rr_sec
    
    qtc_bazett = (qt_sec / np.sqrt(rr_sec)) * 1000.0
    qtc_fridericia = (qt_sec / (rr_sec ** (1/3))) * 1000.0
    qtc_framingham = (qt_sec + 0.154 * (1.0 - rr_sec)) * 1000.0
    qtc_hodges = qt_ms + 1.75 * (hr - 60.0)
    
    return {
        "Heart Rate (bpm)": round(hr, 1),
        "Uncorrected QT (ms)": round(qt_ms, 1),
        "QTc Bazett (ms)": round(qtc_bazett, 1),
        "QTc Fridericia (ms)": round(qtc_fridericia, 1),
        "QTc Framingham (ms)": round(qtc_framingham, 1),
        "QTc Hodges (ms)": round(qtc_hodges, 1)
    }

# Compare QTc corrections at Normal HR (70 bpm) vs Tachycardia (110 bpm)
normal_hr_stats = calculate_qtc_variations(qt_ms=400, rr_ms=857) # 70 bpm
tachy_hr_stats = calculate_qtc_variations(qt_ms=340, rr_ms=545)  # 110 bpm

print("==================================================")
print("Normal Heart Rate (70 bpm) QTc Comparison:")
print(normal_hr_stats)
print("\nTachycardic Heart Rate (110 bpm) QTc Comparison:")
print(tachy_hr_stats)
print("==================================================")

Normal Heart Rate (70 bpm) QTc Comparison:
{'Heart Rate (bpm)': 70.0, 'Uncorrected QT (ms)': 400, 'QTc Bazett (ms)': np.float64(432.1), 'QTc Fridericia (ms)': 421.1, 'QTc Framingham (ms)': 422.0, 'QTc Hodges (ms)': 417.5}

Tachycardic Heart Rate (110 bpm) QTc Comparison:
{'Heart Rate (bpm)': 110.1, 'Uncorrected QT (ms)': 340, 'QTc Bazett (ms)': np.float64(460.6), 'QTc Fridericia (ms)': 416.2, 'QTc Framingham (ms)': 410.1, 'QTc Hodges (ms)': 427.7}
